# Begin

In [1]:
# @launchit.collected

In [2]:
import os # @launchit.collect
import sys # @launchit.collect
import copy
from collections import namedtuple, defaultdict, Counter # @launchit.collect
import itertools
import json
import datetime
import pprint
from functools import cache
import re
import pickle
from unittest.mock import Mock
import dataclasses # @launchit.collect
from dataclasses import dataclass # @launchit.collect
import IPython
from enum import Flag, StrEnum, auto # @launchit.collect
import multiprocessing as mp

from tqdm.notebook import tqdm

import lark 

import numpy as np
import einops
import matplotlib.pyplot as plt
import scipy.signal as signal
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import KFold

import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch.utils.data import DataLoader
from torch.distributions.categorical import Categorical
from torchvision import datasets

import optuna 
from optuna.storages import JournalStorage 
from optuna.storages.journal import JournalFileBackend 
from optuna.trial import TrialState

project_root_path = '${PROJECT_ROOT_PATH}' # @launchit.collect
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

sys.path.append('.')
sys.path.append(os.path.join(project_root_path, 'lib')) # @launchit.collect
from utils import * # @launchit.collect
from logging_utils import *
from model_registry import *
import launchit # @launchit.disable
import optuna_multiprocessing  # @launchit.collect_2
from metrics_collector import RmqSummaryWriter
from autoincrement import Autoincrement
from torch_helpers import *
from hp_utils import *
from model_units import *

# Setup

In [3]:
LOG = Logging.get()
RNG = np.random.default_rng()
METRICS_SUITE = defaultdict(list)

class ExecMode(StrEnum):
    MASTER_NOTEBOOK = auto()
    LAUNCH_NOTEBOOK = auto()
    LAUNCH_MODULE = auto()

CONFIG = namedtuple('Config', 
                    'project_root_path, project_root_uri, subproject_path, data_path, mnist_path, private_data_path, run_path, ' + 
                    'self_fname, self_name, ' +
                    'subproject_name,' +
                    'is_cuda, cuda_device, exec_mode, is_interactive')(
    project_root_path=project_root_path,
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    subproject_path=os.path.abspath('.'),
    data_path=os.path.join(project_root_path, 'data'),
    mnist_path=os.path.join(project_root_path, 'data', 'mnist'),
    private_data_path=None,
    run_path='',
    self_fname='',
    self_name='',
    subproject_name='',
    is_cuda=torch.cuda.is_available(),
    cuda_device='cuda' if torch.cuda.is_available() else 'cpu',
    exec_mode=ExecMode.MASTER_NOTEBOOK,
    is_interactive=True,
)

if IPython.get_ipython() is None:
    module_fname = __file__
    module_basename = os.path.basename(module_fname)
    module_name, _ = os.path.splitext(module_basename)
    
    CONFIG = CONFIG._replace(self_fname=module_fname, self_name=module_name)
    CONFIG = CONFIG._replace(exec_mode=ExecMode.LAUNCH_MODULE)
else:
    with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
        notebook_fname = json.load(cf)['jupyter_session']
        notebook_basename = os.path.basename(notebook_fname)
        notebook_name, notebook_ext = os.path.splitext(notebook_basename)
    
        m = re.match(r'(\w+)-Copy\d+$', notebook_name)
    
        if m: notebook_name = m.group(1) # e.g. Cuml is used to be launched from the copy of the notebook

        CONFIG = CONFIG._replace(self_fname=notebook_fname, self_name=notebook_name)
        
        is_launch = re.match(r'\w+-launch\d+$', notebook_name) is not None
        CONFIG = CONFIG._replace(exec_mode=ExecMode.MASTER_NOTEBOOK if not is_launch else ExecMode.LAUNCH_NOTEBOOK)

CONFIG = CONFIG._replace(is_interactive=CONFIG.exec_mode != ExecMode.LAUNCH_MODULE)

LOG.app_name = CONFIG.self_name
LOG.enable('syslog', not CONFIG.is_interactive)
LOG.enable('stdout', CONFIG.is_interactive)

CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.self_fname)))
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG = CONFIG._replace(private_data_path=os.path.join(CONFIG.data_path, CONFIG.subproject_name))
LOG(f'CONFIG=\n{pprint.pformat(CONFIG._asdict(), sort_dicts=False)}\n', when=CONFIG.is_interactive)
LOG(f'CONFIG={CONFIG._asdict()}', when=not CONFIG.is_interactive)

os.makedirs(CONFIG.private_data_path, exist_ok=True)
os.makedirs(CONFIG.run_path, exist_ok=True)

CONFIG=
{'project_root_path': '/home/misha/dev/mine/neurovision',
 'project_root_uri': 'com.develorium.neurovision',
 'subproject_path': '/home/misha/dev/mine/neurovision/13_gan',
 'data_path': '/home/misha/dev/mine/neurovision/data',
 'mnist_path': '/home/misha/dev/mine/neurovision/data/mnist',
 'private_data_path': '/home/misha/dev/mine/neurovision/data/13_gan',
 'run_path': '/home/misha/dev/mine/neurovision/run/13_gan',
 'self_fname': '/home/misha/dev/mine/neurovision/13_gan/gen_image_gan_02.ipynb',
 'self_name': 'gen_image_gan_02',
 'subproject_name': '13_gan',
 'is_cuda': True,
 'cuda_device': 'cuda',
 'exec_mode': <ExecMode.MASTER_NOTEBOOK: 'master_notebook'>,
 'is_interactive': True}



In [4]:
# @launchit.disable
# @launchit.collect
class LaunchGoal(StrEnum):
    UNSPECIFIED = auto()
    TRAIN_MODEL = auto()

class LaunchGoalObj(namedtuple('LaunchGoalObj', 'goal, model_group_uri, model_name, model_version, model_main_asset_fname')):
    def matches(self, *g):
        if CONFIG.exec_mode == ExecMode.MASTER_NOTEBOOK:
            return True
            
        if g and isinstance(g[0], list):
            assert len(g) == 1
            return self.goal in g[0]

        return self.goal in g

LAUNCH_GOAL = LaunchGoalObj(
    goal=LangUtils.from_str(LaunchGoal, '${LAUNCH_GOAL}', LaunchGoal.UNSPECIFIED),
    model_group_uri='${MODEL_GROUP_URI}',
    model_name='${MODEL_NAME}',
    model_version=LangUtils.from_str(int, '${MODEL_VERSION}', 0),
    model_main_asset_fname='${LAUNCHIT_FNAME}',
)
# @launchit.stop

LAUNCH_GOAL = LAUNCH_GOAL._replace(goal=LaunchGoal.UNSPECIFIED)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_name=CONFIG.self_name)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_version=0)
LAUNCH_GOAL = LAUNCH_GOAL._replace(model_main_asset_fname=CONFIG.self_fname)
# @launchit.stop

LOG(f'LAUNCH_GOAL=\n{pprint.pformat(LAUNCH_GOAL._asdict(), sort_dicts=False)}', when=CONFIG.is_interactive)
LOG(f'LAUNCH_GOAL={LAUNCH_GOAL._asdict()}', when=not CONFIG.is_interactive)

LAUNCH_GOAL=
{'goal': <LaunchGoal.UNSPECIFIED: 'unspecified'>,
 'model_group_uri': 'com.develorium.neurovision.13_gan',
 'model_name': 'gen_image_gan_02',
 'model_version': 0,
 'model_main_asset_fname': '/home/misha/dev/mine/neurovision/13_gan/gen_image_gan_02.ipynb'}


# Hyperparameters

In [5]:
# @launchit.disable
# @launchit.collect
@dataclass
class Hyperparameters:
    random_seed: int = None
    # input data params
    image_shape: tuple = (28, 28)
    images_preprocessing: str = None
    # model structure params
    discriminator_model_units: list[str] = None
    generator_noise_size: int = None
    generator_noise_source: str = None
    generator_model_units: list[str] = None
    # training params
    batch_size: int = None
    epochs_count: int = None
    optimizer: str = None
    discriminator_learn_rate: float | str = None
    generator_learn_rate: float | str = None

    @staticmethod
    def from_dict(d):
        hp = Hyperparameters(**d)
        return hp

    def _asdict(self):
        return dataclasses.asdict(self)

HP = Hyperparameters()
HP.random_seed = 1


Замачения по поводу гиперпараметров:
1) модель очень чувствительна к `batch_size`. При `batch_size=64` наблюдается сходимость к приемлемому `l1_probas`, при более высоких значения - непредсказуемо и нестабильно. Это сильно контрастирует с прошлыми экспериментами, где можно было `batch_size=1000` иметь
2) также модель очень чувствительна к `random_seed`. При `random_seed=1` наблюдается сходимость к приемлемому `l1_probas`, при `random_seed=42` сходимость срывается с катушек. Другие значения не испытывал, но вот такое кардинальное различие говорит о том, что начальное состояние прям очень важно.
3) удивительно, но всегда хорошо работающий `learn_rate=0.001, plateau()` показал себя плохо, лучше оставить просто `learn_rate=0.001`

Вероятно, у модели есть какие-то проблемы с capacity, и неудачные начальные значения отравляют жизнь.

# Launch

## new_model_registry

In [6]:
def new_model_registry(is_real=None, group_uri=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        mr = Mock()
        mr.register_model.return_value = 0
        return mr
        
    return ModelRegistry(LAUNCH_GOAL.model_group_uri if group_uri is None else group_uri)

## new_summary_writer

In [7]:
def new_summary_writer(log_dir, is_real=None):
    is_launch = CONFIG.exec_mode in [ExecMode.LAUNCH_NOTEBOOK, ExecMode.LAUNCH_MODULE]
    is_real = is_real if is_real is not None else is_launch

    if not is_real:
        sw = Mock()
        sw.flush.side_effect = sw.reset_mock # to get rid of all recorded call_args_list, which might be heavy (e.g. add_figure)
        return sw
    
    return RmqSummaryWriter(log_dir)

## Bootstrap

In [8]:
optuna_trial = optuna_multiprocessing.get_trial()
optuna_trial_subdir_name = ''

if optuna_trial is not None:
    optuna_trial.set_user_attr('MODEL_VERSION', LAUNCH_GOAL.model_version)
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    optuna_trial_subdir_name = f'opt_{study_serial}'
    LOG(f'Optuna {optuna_trial.number=}, {optuna_trial.user_attrs=}')

LOG(f'HP={HP._asdict()}', when=not CONFIG.is_interactive)
    
if HP.random_seed is not None:
    torch.manual_seed(HP.random_seed)
    RNG = np.random.default_rng(HP.random_seed)    
    LOG(f'Random seed={HP.random_seed}')

if True or LAUNCH_GOAL.goal != LaunchGoal.TRAIN_MODEL_AFTER_CV:
    model_registry = new_model_registry()
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, CONFIG.self_fname, replace=True)
        
    meta = dict(
        optuna_trial_number=getattr(optuna_trial, 'number', None),
        hypers=HP._asdict(), 
        config=CONFIG._asdict(), 
    )
    
    with io.StringIO() as b:
        json.dump(meta, b)
        model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='meta', replace=True)

summary_log_dir = LAUNCH_GOAL.model_name
summary_log_dir = os.path.join(summary_log_dir, optuna_trial_subdir_name) if optuna_trial_subdir_name != '' else summary_log_dir 
summary_log_dir = os.path.join(summary_log_dir, str(LAUNCH_GOAL.model_version))
LOG(f'Tensorboard run={summary_log_dir}')
summary_writer = new_summary_writer(log_dir=summary_log_dir)
summary_writer.add_text('hypers', pprint.pformat(HP._asdict(), sort_dicts=False), 1)
summary_writer.add_text('config', pprint.pformat(CONFIG._asdict(), sort_dicts=False), 1)

Random seed=1
Tensorboard run=gen_image_gan_02/0


<Mock name='mock.add_text()' id='139442368588608'>

# Images

In [9]:
# @launchit.disable
# @launchit.collect_1
HP.images_preprocessing = 'MIN_MAX(-1,+1)' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN

## get_mnist_images

In [10]:
def get_mnist_images(subdataset='TRAIN'):
    assert subdataset in ['TRAIN', 'TEST'], f'Unsupported subdataset={subdataset}'
    d = datasets.MNIST(CONFIG.mnist_path, train=subdataset=='TRAIN', download=True)
    images = d.data.numpy()
    images = images.astype('float32')
    image_labels = d.targets
    return images, image_labels

## UninormScaler

In [11]:
class UninormScaler:
    def __init__(self, divisor=255.0, feature_range=None):
        self.divisor = divisor
        self.feature_range = feature_range

        if self.feature_range is not None:
            assert isinstance(self.feature_range, tuple)
            assert len(self.feature_range) == 2
            assert self.feature_range[0] < self.feature_range[1]
        
    def fit_transform(self, images):
        return self.transform(images)

    def transform(self, images):
        images = images / self.divisor
        
        if self.feature_range is not None:
            d = self.feature_range[1] - self.feature_range[0]
            assert d > 0
            images -= 0.5
            images *= d
            
        return images

## SampleWiseScaler

In [12]:
# https://gist.github.com/kocherovms/ca352c30fe3eea0f155d4862ddde6e3a for tests and breakdown
class SampleWiseScaler:
    def fit_transform(self, images):
        return self.transform(images)

    # Images are expected to be in raveled (flattened) mode => only last dim is taken into account
    def transform(self, images):
        shape = images.shape
        images = images.reshape(-1, images.shape[-1]) # get rid of all dimensions except the last one
        means = images.mean(axis=-1)
        stds = images.std(axis=-1)
        images = images.T - means
        images = images / np.where(stds != 0, stds, 1)
        images = images.T
        return images.reshape(shape)

## preprocess_images

In [13]:
def preprocess_images(images, preprocessing_method, scaler=None, is_flattened=True):
    if not is_flattened:
        shape = images.shape
        images = einops.rearrange(images, '... h w -> ... (h w)') # collapse last two dimension
    
    match preprocessing_method:
        case 'UNINORM':
            scaler = UninormScaler(divisor=255) if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'UNINORM(-1,+1)':
            scaler = UninormScaler(divisor=255, feature_range=(-1, +1)) if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'SAMPLE_WISE':
            scaler = SampleWiseScaler() if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'MIN_MAX(-1,+1)':
            scaler = MinMaxScaler(feature_range=(-1, +1)) if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'STANDARDIZE':
            scaler = StandardScaler() if scaler is None else scaler
            images = scaler.fit_transform(images)
        case 'ZCA_WHITEN':
            scaler = StandardScaler(with_std=False) if scaler is None else scaler
            images = scaler.fit_transform(images)
            
            Σ = np.cov(images, rowvar=False)
            u, s, _ = np.linalg.svd(Σ)
            images = (u @ np.diag(1.0 / np.sqrt(s + 1e-6)) @ u.T @ images.T).T
        case 'ZCA_HFR30_WHITEN': # HFR30 - Remove 30% of High Frequencies
            scaler = StandardScaler(with_std=False) if scaler is None else scaler
            images = scaler.fit_transform(images)
    
            Σ = np.cov(images, rowvar=False)
            eigvals, eigvecs = np.linalg.eig(Σ)
            eigvals_order = np.argsort(-eigvals)
            wipeout_inds = eigvals_order[int(len(eigvals_order) * (1 - 0.3)):]
            eigvals_w = eigvals.copy()
            eigvals_w[wipeout_inds] = 0
            
            R, S = eigvecs, np.diag(np.sqrt(eigvals_w)) # R - rotation matrix, S - scale matrix
            S_inv = np.reciprocal(S, out=np.zeros_like(S), where=(S != 0))
            R_inv = R.T
            W = R @ S_inv @ R_inv  # equiv. to: R @ np.eye(len(S_inv)) @ S_inv @ R_inv
            images = (W @ images.T).T
        case 'NONE':
            pass
        case _:
            assert False, f'Unsupported preprocessing_method={preprocessing_method}'

    if not is_flattened:
        images = images.reshape(shape)
        
    return images, scaler

# Dataset

## create_dataset_for_unsupervised_pretraining

In [14]:
def create_dataset_for_unsupervised_pretraining():
    images, _ = get_mnist_images('TRAIN')
    assert images.shape[-2:] == HP.image_shape
    images, _ = preprocess_images(images, HP.images_preprocessing, is_flattened=False)
    images = torch.Tensor(images)
    images = images.contiguous() # force dense memory layout (speeds up DataLoader x2 in case of any transposes within preprocess_images)
    return images

# Model

## DiscriminatorModel

In [15]:
class DiscriminatorModel(nn.Module):
    def __init__(self, hp):
        super().__init__()
        self.units = nn.Sequential()

        for params in hp_parse_model_units(hp.discriminator_model_units):
            assert isinstance(params, Conv2dModelUnitParams)
            unit = create_unit(params)
            self.units.append(unit)

    def forward(self, inp):
        res = self.units(inp)
        return einops.rearrange(res, 'b c h w -> (b c h w) 1')

In [16]:
# class Discriminator(nn.Module):
#     def __init__(self, n_filters):
#         super().__init__()
#         self.network = nn.Sequential(
#             nn.Conv2d(1, n_filters, 4, 2, 1, bias=False),
#             nn.LeakyReLU(0.2),

#             nn.Conv2d(n_filters, n_filters*2, 4, 2, 1, bias=False),
#             nn.BatchNorm2d(n_filters * 2),
#             nn.LeakyReLU(0.2),

#             nn.Conv2d(n_filters*2, n_filters*4, 3, 2, 1, bias=False),
#             nn.BatchNorm2d(n_filters*4),
#             nn.LeakyReLU(0.2),

#             nn.Conv2d(n_filters*4, 1, 4, 1, 0, bias=False),
#             nn.Sigmoid())
        
#     def forward(self, input):
#         output = self.network(input)
#         return output.view(-1, 1).squeeze(0)

# Discriminator(
#   (network): Sequential(
#     (0): Conv2d(1, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
#     (1): LeakyReLU(negative_slope=0.2)
#     (2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
#     (3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
#     (4): LeakyReLU(negative_slope=0.2)
#     (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
#     (6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
#     (7): LeakyReLU(negative_slope=0.2)
#     (8): Conv2d(128, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
#     (9): Sigmoid()
#   )
# )

In [17]:
# @launchit.disable
model_hp = Hyperparameters(discriminator_model_units=[
    'Conv2d: conv(1->32(1)x4, stride=2, padding=1)->LeakyReLU(0.2)',
    'Conv2d: conv(32->64(32)x4, stride=2, padding=1)->BatchNorm2d(64)->LeakyReLU(0.2)',
    'Conv2d: conv(64->128(64)x3, stride=2, padding=1)->BatchNorm2d(128)->LeakyReLU(0.2)',
    'Conv2d: conv(128->1(128)x4, stride=1, padding=0)->Sigmoid',
])
model = DiscriminatorModel(model_hp)
probe_tensor = torch.ones(10, 1, *model_hp.image_shape)
res = model(probe_tensor)
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Input shape={probe_tensor.shape}')
print(f'Output shape={res.shape}')

DiscriminatorModel(
  (units): Sequential(
    (0): Conv2dModelUnit(
      (conv): Conv2d(1, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (nonlinearity): LeakyReLU(negative_slope=0.2)
    )
    (1): Conv2dModelUnit(
      (conv): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (batch_norm2d): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (nonlinearity): LeakyReLU(negative_slope=0.2)
    )
    (2): Conv2dModelUnit(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (batch_norm2d): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (nonlinearity): LeakyReLU(negative_slope=0.2)
    )
    (3): Conv2dModelUnit(
      (conv): Conv2d(128, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
      (nonlinearity): Sigmoid()
    )
  )
)
Parameters count=109440
Input shape=torch.Size([10, 1, 28, 28])
Output shape=t

## GeneratorModel

In [18]:
class GeneratorModel(nn.Module):
    def __init__(self, hp):
        super().__init__()
        self.noise_size = hp.generator_noise_size
        self.noise_source = hp.generator_noise_source
        self.units = nn.Sequential()

        for params in hp_parse_model_units(hp.generator_model_units):
            assert isinstance(params, Conv2dModelUnitParams)
            unit = create_unit(params)
            self.units.append(unit)

    def forward(self, inp):
        return self.units(inp)

    # Unlike create_noise in linear based GAN here create_noise returns multichannel (e.g. 100 channels) image consisting of 1 pixel
    def create_noise(self, batch_size):
        match self.noise_source:
            case 'uniform':
                return torch.rand(batch_size, self.noise_size, 1, 1) * 2 - 1 # rvs from range [-1, +1]
            case 'normal':
                return torch.randn(batch_size, self.noise_size, 1, 1)
            case _:
                assert False, f'Unsupported {self.noise_source=}'

In [19]:
# def make_generator_network(input_size, n_filters):
#     model = nn.Sequential(
#         nn.ConvTranspose2d(input_size, n_filters*4, 4, 1, 0, 
#                            bias=False),
#         nn.BatchNorm2d(n_filters*4),
#         nn.LeakyReLU(0.2),

#         nn.ConvTranspose2d(n_filters*4, n_filters*2, 3, 2, 1, bias=False),
#         nn.BatchNorm2d(n_filters*2),
#         nn.LeakyReLU(0.2),

#         nn.ConvTranspose2d(n_filters*2, n_filters, 4, 2, 1, bias=False),
#         nn.BatchNorm2d(n_filters),
#         nn.LeakyReLU(0.2),

#         nn.ConvTranspose2d(n_filters, 1, 4, 2, 1, bias=False),
#         nn.Tanh())
#     return model
# Sequential(
#   (0): ConvTranspose2d(100, 128, kernel_size=(4, 4), stride=(1, 1), bias=False)
#   (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
#   (2): LeakyReLU(negative_slope=0.2)
#   (3): ConvTranspose2d(128, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
#   (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
#   (5): LeakyReLU(negative_slope=0.2)
#   (6): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
#   (7): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
#   (8): LeakyReLU(negative_slope=0.2)
#   (9): ConvTranspose2d(32, 1, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
#   (10): Tanh()
# )

In [20]:
# @launchit.disable
generator_noise_size = 100
model_hp = Hyperparameters(
    generator_noise_size=generator_noise_size,
    generator_noise_source='uniform',
    generator_model_units=[
        f'ConvTranspose2d: conv({generator_noise_size}->128(100)x4-bias,stride=1,padding=0)->BatchNorm2d(128)->LeakyReLU(0.2)',
        f'ConvTranspose2d: conv(128->64(128)x3-bias,stride=2,padding=1)->BatchNorm2d(64)->LeakyReLU(0.2)',
        f'ConvTranspose2d: conv(64->32(64)x4-bias,stride=2,padding=1)->BatchNorm2d(32)->LeakyReLU(0.2)',
        f'ConvTranspose2d: conv(32->1(32)x4-bias,stride=2,padding=1)->Tanh',
    ]
)
model = GeneratorModel(model_hp)
probe_tensor = model.create_noise(3)
res = model(probe_tensor)
assert res.shape[-2:] == HP.image_shape, res.shape
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')
print(f'Input shape={probe_tensor.shape}')
print(f'Output shape={res.shape}')

GeneratorModel(
  (units): Sequential(
    (0): Conv2dModelUnit(
      (conv): ConvTranspose2d(100, 128, kernel_size=(4, 4), stride=(1, 1), bias=False)
      (batch_norm2d): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (nonlinearity): LeakyReLU(negative_slope=0.2)
    )
    (1): Conv2dModelUnit(
      (conv): ConvTranspose2d(128, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (batch_norm2d): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (nonlinearity): LeakyReLU(negative_slope=0.2)
    )
    (2): Conv2dModelUnit(
      (conv): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
      (batch_norm2d): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (nonlinearity): LeakyReLU(negative_slope=0.2)
    )
    (3): Conv2dModelUnit(
      (conv): ConvTranspose2d(32, 1, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1)

## MainModel

In [21]:
class MainModel(nn.Module):
    def __init__(self, hp):
        super().__init__()
        self.d = DiscriminatorModel(hp)
        self.g = GeneratorModel(hp)

    def forward(self, inp):
        assert False, 'Should not be called'

In [22]:
# @launchit.disable
generator_noise_size = 100
model_hp = Hyperparameters(
    discriminator_model_units=[
        'Conv2d: conv(1->32(1)x4, stride=2, padding=1)->LeakyReLU(0.2)',
        'Conv2d: conv(32->64(32)x4, stride=2, padding=1)->BatchNorm2d(64)->LeakyReLU(0.2)',
        'Conv2d: conv(64->128(64)x3, stride=2, padding=1)->BatchNorm2d(128)->LeakyReLU(0.2)',
        'Conv2d: conv(128->1(128)x4, stride=1, padding=0)->Sigmoid',
    ],
    generator_noise_size=generator_noise_size,
    generator_noise_source='uniform',
    generator_model_units=[
        f'ConvTranspose2d: conv({generator_noise_size}->128(100)x4-bias,stride=1,padding=0)->BatchNorm2d(128)->LeakyReLU(0.2)',
        f'ConvTranspose2d: conv(128->64(128)x3-bias,stride=2,padding=1)->BatchNorm2d(64)->LeakyReLU(0.2)',
        f'ConvTranspose2d: conv(64->32(64)x4-bias,stride=2,padding=1)->BatchNorm2d(32)->LeakyReLU(0.2)',
        f'ConvTranspose2d: conv(32->1(32)x4-bias,stride=2,padding=1)->Tanh',
    ]
)
model = MainModel(model_hp)
print(model)
print(f'Parameters count={sum([p.numel() for p in model.parameters()]):}')

MainModel(
  (d): DiscriminatorModel(
    (units): Sequential(
      (0): Conv2dModelUnit(
        (conv): Conv2d(1, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
        (nonlinearity): LeakyReLU(negative_slope=0.2)
      )
      (1): Conv2dModelUnit(
        (conv): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
        (batch_norm2d): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (nonlinearity): LeakyReLU(negative_slope=0.2)
      )
      (2): Conv2dModelUnit(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (batch_norm2d): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (nonlinearity): LeakyReLU(negative_slope=0.2)
      )
      (3): Conv2dModelUnit(
        (conv): Conv2d(128, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
        (nonlinearity): Sigmoid()
      )
    )
  )
  (g): GeneratorMo

# LaunchGoal.TRAIN_MODEL

## Configure

In [43]:
# @launchit.disable
# @launchit.collect_1
HP.discriminator_model_units = [
    'Conv2d: conv(1->32(1)x4, stride=2, padding=1)->LeakyReLU(0.2)',
    'Conv2d: conv(32->64(32)x4, stride=2, padding=1)->BatchNorm2d(64)->LeakyReLU(0.2)',
    'Conv2d: conv(64->128(64)x3, stride=2, padding=1)->BatchNorm2d(128)->LeakyReLU(0.2)',
    'Conv2d: conv(128->1(128)x4, stride=1, padding=0)->Sigmoid',
]
HP.generator_noise_size = 100
HP.generator_noise_source = 'uniform'
HP.generator_model_units = [
    f'ConvTranspose2d: conv({HP.generator_noise_size}->128(100)x4-bias,stride=1,padding=0)->BatchNorm2d(128)->LeakyReLU(0.2)',
    f'ConvTranspose2d: conv(128->64(128)x3-bias,stride=2,padding=1)->BatchNorm2d(64)->LeakyReLU(0.2)',
    f'ConvTranspose2d: conv(64->32(64)x4-bias,stride=2,padding=1)->BatchNorm2d(32)->LeakyReLU(0.2)',
    f'ConvTranspose2d: conv(32->1(32)x4-bias,stride=2,padding=1)->Tanh',
]
HP.batch_size = 100
HP.epochs_count = 1
HP.optimizer = 'Adam'
HP.discriminator_learn_rate = 0.0002
HP.generator_learn_rate = 0.0003
# @launchit.stop
LOG(pprint.pformat(HP._asdict(), sort_dicts=False), when=CONFIG.is_interactive)

{'random_seed': 1,
 'image_shape': (28, 28),
 'images_preprocessing': 'MIN_MAX(-1,+1)',
 'discriminator_model_units': ['Conv2d: conv(1->32(1)x4, stride=2, '
                               'padding=1)->LeakyReLU(0.2)',
                               'Conv2d: conv(32->64(32)x4, stride=2, '
                               'padding=1)->BatchNorm2d(64)->LeakyReLU(0.2)',
                               'Conv2d: conv(64->128(64)x3, stride=2, '
                               'padding=1)->BatchNorm2d(128)->LeakyReLU(0.2)',
                               'Conv2d: conv(128->1(128)x4, stride=1, '
                               'padding=0)->Sigmoid'],
 'generator_noise_size': 100,
 'generator_noise_source': 'uniform',
 'generator_model_units': ['ConvTranspose2d: '
                           'conv(100->128(100)x4-bias,stride=1,padding=0)->BatchNorm2d(128)->LeakyReLU(0.2)',
                           'ConvTranspose2d: '
                           'conv(128->64(128)x3-bias,stride=2,padding=1)->BatchNorm

## Create

In [44]:
dataset = create_dataset_for_unsupervised_pretraining()
data_loader = DataLoader(dataset, batch_size=HP.batch_size, shuffle=True)
model = MainModel(HP)
model = model.to(device=CONFIG.cuda_device)
loss_fn = nn.BCELoss()
OptimizerStuff = namedtuple('OptimizerStuff', 'optimizer lr_scheduler')
optimizers = {}

for key, submodel, learn_rate in zip(('d', 'g'), (model.d, model.g), (HP.discriminator_learn_rate, HP.generator_learn_rate)):
    lr_params = hp_parse_learn_rate(learn_rate)
    optimizer = getattr(torch.optim, HP.optimizer)(submodel.parameters(), lr=lr_params.learn_rate)
    lr_scheduler = LrSchedulerWrapper(optimizer, lr_params)
    optimizers[key] = OptimizerStuff(optimizer, lr_scheduler)

In [45]:
model

MainModel(
  (d): DiscriminatorModel(
    (units): Sequential(
      (0): Conv2dModelUnit(
        (conv): Conv2d(1, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
        (nonlinearity): LeakyReLU(negative_slope=0.2)
      )
      (1): Conv2dModelUnit(
        (conv): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
        (batch_norm2d): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (nonlinearity): LeakyReLU(negative_slope=0.2)
      )
      (2): Conv2dModelUnit(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (batch_norm2d): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (nonlinearity): LeakyReLU(negative_slope=0.2)
      )
      (3): Conv2dModelUnit(
        (conv): Conv2d(128, 1, kernel_size=(4, 4), stride=(1, 1), bias=False)
        (nonlinearity): Sigmoid()
      )
    )
  )
  (g): GeneratorMo

In [46]:
optimizers

{'d': OptimizerStuff(optimizer=Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     decoupled_weight_decay: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0003
     maximize: False
     weight_decay: 0
 ), lr_scheduler=<torch_helpers.LrSchedulerWrapper object at 0x702cff9573b0>),
 'g': OptimizerStuff(optimizer=Adam (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     decoupled_weight_decay: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0002
     maximize: False
     weight_decay: 0
 ), lr_scheduler=<torch_helpers.LrSchedulerWrapper object at 0x702cf8bea120>)}

## train_discriminator

In [47]:
def train_discriminator(model, optimizers, epoch, batch):
    batch_size = batch.shape[0]
    optimizer = optimizers['d'].optimizer
    optimizer.zero_grad()
    
    # Real images
    real_labels = torch.ones(batch_size, 1).to(CONFIG.cuda_device)
    real_probas = model.d(batch)
    real_loss = loss_fn(real_probas, real_labels)

    # Generated images
    generated_labels = torch.zeros(batch_size, 1).to(CONFIG.cuda_device)
    noise = model.g.create_noise(batch_size).to(CONFIG.cuda_device)
    generated = model.g(noise)
    generated_probas = model.d(generated)
    generated_loss = loss_fn(generated_probas, generated_labels)
    # Use both terms of V function
    loss = real_loss + generated_loss

    if epoch > 0:
        loss.backward()
        optimizer.step()
  
    return loss.item(), real_probas.detach(), generated_probas.detach()

In [48]:
# train_discriminator(model, optimizers, 0, einops.rearrange(next(iter(data_loader)), 'b h w -> b 1 h w'))

## train_generator

In [49]:
def train_generator(model, optimizers, epoch, batch):
    batch_size = batch.shape[0]
    optimizer = optimizers['g'].optimizer
    optimizer.zero_grad()

    # We want generator to generate images which discriminator treats as real, 
    # as such demand all labels to be True (1) forcing generator to improve its behavior
    wanted_labels = torch.ones(batch_size, 1).to(CONFIG.cuda_device)
    noise = model.g.create_noise(batch_size).to(CONFIG.cuda_device)
    generated = model.g(noise)
    generated_probas = model.d(generated)
    generated_loss = loss_fn(generated_probas, wanted_labels)
    # Use only second term of V function
    loss = 0 + generated_loss

    if epoch > 0:
        loss.backward()
        optimizer.step()
        
    return loss.item()

In [50]:
# train_generator(model, optimizers, 0, einops.rearrange(next(iter(data_loader)), 'b h w -> b 1 h w'))

## Train

In [52]:
for epoch in tqdm(range(0, HP.epochs_count + 1), 'Epoch', disable=not CONFIG.is_interactive):
    with LOG.auto_prefix('EPOCH', epoch):
        epoch_sums = defaultdict(int)
        epoch_denom = 0
        
        for batch in tqdm(data_loader, 'Batch', disable=not CONFIG.is_interactive, leave=False):
            batch = einops.rearrange(batch, 'b h w -> b 1 h w')
            batch = batch.to(CONFIG.cuda_device)
            
            d_loss, d_real_probas, d_generated_probas = train_discriminator(model, optimizers, epoch, batch)
            g_loss = train_generator(model, optimizers, epoch, batch)

            epoch_sums['loss_train'] += (d_loss + g_loss) * len(batch)
            epoch_sums['loss_d'] += d_loss * len(batch)
            epoch_sums['loss_g'] += g_loss * len(batch)
            epoch_sums['probas_real'] += d_real_probas.sum().item() # ideally this should reach 0.5 together with generated_probas=0.5
            epoch_sums['probas_generated'] += d_generated_probas.sum().item() # ideally this should reach 0.5 together with real_probas=0.5
            epoch_sums['probas_l1'] += torch.abs(d_real_probas - d_generated_probas).sum().item()
            epoch_denom += len(batch)
    
        assert epoch_denom > 0

        for k, v in epoch_sums.items():
            v = v / epoch_denom
            summary_writer.add_scalar(k, v, epoch)
            METRICS_SUITE[k].append(v)
            LOG(f'{k}={v:.2f}', when=not CONFIG.is_interactive)

            if k == 'loss_d':
                optimizers['d'].lr_scheduler.step(v)
            elif k == 'loss_g':
                optimizers['g'].lr_scheduler.step(v)

        if optuna_trial is not None:
            # https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html#optuna.trial.Trial.report:
            # > If this method is called multiple times at the same step in a trial, the reported value only the first time is stored 
            # > and the reported values from the second time are ignored.
            # In other words calling report for fold other than the first one does nothing except producing tons of warnings in console. 
            # As such only the first fold is indicative
            probas_l1 = epoch_sums['probas_l1'] / epoch_denom
            optuna_trial.report(probas_l1, epoch) 
    
            if optuna_trial.should_prune():
                # Despite written in docs OPTUNA_TRIAL.should_prune is not idempotent - consequent calls could lead
                # to different responses. Perhapse this is due to influence of concurrent trials running which could change
                # pruner decision. As such cache pruning result so it's immutable
                optuna_trial.set_user_attr('IS_PRUNED', True)
                LOG(f'Optuna pruning condition encountered. Stopping training')
                break

        if epoch % 10 == 0:
            if epoch == 0:
                noise_for_examples = model.g.create_noise(10).to(CONFIG.cuda_device)
                
            model.eval()
            
            with torch.no_grad():
                generated = model.g(noise_for_examples).cpu()
                generated = einops.rearrange(generated, 'b 1 h w -> b h w')
                fig, axes = plt.subplots(1, len(generated))
                fig.set_figwidth(16)
                fig.set_figheight(1.8)
                for ax in axes.ravel(): ax.set_axis_off()

                for ind, ax in enumerate(axes):
                    ax.imshow(generated[ind])

                fig.tight_layout()
                summary_writer.add_figure(f'examples', fig, epoch, close=False) 
                plt.close(fig) # manual closing because summar_writer may be in Mock mode
                LOG('Figure "examples" uploaded', when=not CONFIG.is_interactive)

            model.train()
    
        summary_writer.flush()

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

Batch:   0%|          | 0/600 [00:00<?, ?it/s]

Batch:   0%|          | 0/600 [00:00<?, ?it/s]

## Visualize

In [18]:
# @launchit.disable
noise_for_examples = model.g.create_noise(10).to(CONFIG.cuda_device)
model.eval()
            
with torch.no_grad():
    generated = model.g(noise_for_examples).cpu()
    generated = einops.rearrange(generated, 'b 1 h w -> b h w')
    fig, axes = plt.subplots(1, len(generated))
    fig.set_figwidth(16)
    fig.set_figheight(1.8)
    for ax in axes.ravel(): ax.set_axis_off()

    for ind, ax in enumerate(axes):
        ax.imshow(generated[ind])

    fig.tight_layout()

NameError: name 'model' is not defined

## Save model

In [ ]:
# @launchit.disable_2
model_registry = new_model_registry()

with io.BytesIO() as b:
    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'hypers': HP._asdict(),
    }, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='pt', asset_classifier='main', replace=True)

with io.StringIO() as b:
    json.dump(METRICS_SUITE, b)
    model_registry.attach_asset(LAUNCH_GOAL.model_name, LAUNCH_GOAL.model_version, b, asset_ext='json', asset_classifier='metrics', replace=True)

## Save Optuna trial result

In [ ]:
# @launchit.disable_1
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None and LAUNCH_GOAL.goal == LaunchGoal.TRAIN_MODEL:
    if 'IS_PRUNED' in optuna_trial.user_attrs:
        raise optuna.exceptions.TrialPruned()

    if not METRICS_SUITE:
        LOG(f'Empty metrics suite. Cancelling trial')
        raise Exception('Empty metrics suite')
    else:
        last_probas_l1 = METRICS_SUITE['probas_l1'][-1]
        optuna_multiprocessing.save_trial_result(last_probas_l1)
        LOG(f'Trial objective result: {last_probas_l1=}')

# Launch creation

## TRAIN_MODEL

In [15]:
# @launchit.disable
launchit_t0 = time.time()

In [17]:
# @launchit.disable
launchit_interval = time.time() - launchit_t0

if launchit_interval > 0.05:
    model_version = int(Autoincrement.get(f'{LAUNCH_GOAL.model_group_uri}.{LAUNCH_GOAL.model_name}'))
    assert model_version > 0, model_version
    model_registry_obj = new_model_registry(is_real=True)
    model_registry_obj.register_model(LAUNCH_GOAL.model_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    expandvars = dict(
        PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
        MODEL_NAME=LAUNCH_GOAL.model_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL=LaunchGoal.TRAIN_MODEL,
    )
    launch_notebook_fname = launchit.launchit(CONFIG.self_fname, launch_serial=model_version, expandvars=expandvars, collect_inds=[1], disable_inds=[])
    LOG(f'Created launch notebook "{launch_notebook_fname}"')
else:
    LOG('Skip launchit due to mass "Run Cells"')

Model instance registered, version=2
Creating /home/misha/dev/mine/neurovision/13_gan/gen_image_gan_02-launch2.ipynb
Created launch notebook "/home/misha/dev/mine/neurovision/13_gan/gen_image_gan_02-launch2.ipynb"


## Optuna (model selection)

### Templates

In [26]:
# @launchit.disable
# @launchit.collect_2
optuna_trial = optuna_multiprocessing.get_trial()

if optuna_trial is not None:
    study_serial = optuna_trial.user_attrs['STUDY_SERIAL']
    
    match study_serial:
        case 1:
            HP.random_seed = 1
            HP.images_preprocessing = 'MIN_MAX(-1,+1)' # NONE, UNINORM, SAMPLE_WISE, MIN_MAX, STANDARDIZE, ZCA_WHITEN, ZCA_HFR30_WHITEN
            HP.discriminator_model_units = [
                'Linear(100->LeakyReLU)',
                'Linear(dropout(0.5)->1->Sigmoid)',
            ]
            HP.generator_noise_vector_size = 20
            HP.generator_noise_source = 'uniform'
            HP.generator_model_units = [
                'Linear(100->LeakyReLU)',
                'Linear($output_size->Tanh)',
            ]
            HP.batch_size = optuna_trial.suggest_categorical('batch_size', [64, 100, 200, 300, 500])
            HP.epochs_count = 30
            HP.optimizer = 'Adam'
            HP.learn_rate = optuna_trial.suggest_float('learn_rate', 0.0001, 0.005)
        case _:
            assert False, f'Unsupported {study_serial=}'    

### Unleash

In [80]:
# @launchit.disable
def get_optimize_directions(lg):
    match lg:
        case LaunchGoal.TRAIN_MODEL:
            return ['minimize']
        case _:
            assert False, f'Unsupported {lg=}'

lg = LaunchGoal.TRAIN_MODEL
expandvars = dict(
    PROJECT_ROOT_PATH=CONFIG.project_root_path,
    MODEL_GROUP_URI=LAUNCH_GOAL.model_group_uri,
    MODEL_NAME=LAUNCH_GOAL.model_name,
    LAUNCH_GOAL=lg.value,
)
study_serial = 1
study_name = f'{CONFIG.self_name}_{expandvars['LAUNCH_GOAL']}_{study_serial}'
rop_task = optuna_multiprocessing.RunOptimizationTask(
    app_name=CONFIG.self_name,
    is_stdout_enabled=False,
    notebook_fname=CONFIG.self_fname,
    notebook_name=CONFIG.self_name,
    model_group_uri=LAUNCH_GOAL.model_group_uri,
    model_name=LAUNCH_GOAL.model_name,
    expandvars=expandvars,
    collect_inds=[2],
    disable_inds=[2],
    run_path=CONFIG.run_path,
    study_serial=study_serial,
    study_name=study_name,
    study_fname=os.path.join(CONFIG.run_path, study_name + '.log'),
    optimize_directions=get_optimize_directions(lg),
)
rop_tasks = [rop_task] * 32
mp_ctx = mp.get_context('spawn') # Req-d for CUDA, fork doesn't work within PyTorch

with mp_ctx.Pool(processes=4, maxtasksperchild=1) as pool:  # maxtasksperchild=1 forces fresh process for each trial to spare resources and avoid possible side effects of processe resue
    pool.map(optuna_multiprocessing.run_optimization, rop_tasks)

[I 2026-03-12 21:07:29,662] Using an existing study with name 'gen_image_gan_01_train_model_1' instead of creating a new one.
[I 2026-03-12 21:07:29,673] Using an existing study with name 'gen_image_gan_01_train_model_1' instead of creating a new one.
[I 2026-03-12 21:07:29,675] Using an existing study with name 'gen_image_gan_01_train_model_1' instead of creating a new one.
[I 2026-03-12 21:07:29,687] Using an existing study with name 'gen_image_gan_01_train_model_1' instead of creating a new one.
[I 2026-03-12 21:09:17,842] Trial 2 finished with value: 0.2851074157079061 and parameters: {'batch_size': 300, 'learn_rate': 0.0028800018948288686}. Best is trial 1 with value: 0.25874247487386065.
[I 2026-03-12 21:09:17,961] Using an existing study with name 'gen_image_gan_01_train_model_1' instead of creating a new one.
[I 2026-03-12 21:10:22,838] Trial 6 finished with value: 0.3812264119466146 and parameters: {'batch_size': 500, 'learn_rate': 0.004820241863458476}. Best is trial 1 with v

### Stats

In [81]:
# @launchit.disable
study = optuna.create_study(
    study_name=rop_task.study_name,
    storage=JournalStorage(JournalFileBackend(file_path=rop_task.study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs['MODEL_VERSION']}')
    
    LOG('  Params: ')
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    print(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        print(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        print(f"\tnumber: {trial.number}")
        print(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        print(f"\tparams: {trial.params}")
        print(f"\tvalues: {trial.values}")

[I 2026-03-12 22:05:29,506] Using an existing study with name 'gen_image_gan_01_train_model_1' instead of creating a new one.


Study statistics: 
	Number of finished trials: 34
	Number of pruned trials: 4
	Number of complete trials: 29
Best trial:
	Value: 0.094513529920578
	Model version: 33
  Params: 
		batch_size: 64
		learn_rate: 0.0015989954537675736
